### #1 Attention Mechanism 개념 정리

**1. Query, Key, Value**
정보 검색 시스템에 비유할 수 있으며, 딕셔너리 조회(`dict[key] → value`)의 미분 가능한 소프트 버전입니다.

* **Query (Q):** "내가 지금 찾고 있는 것"을 의미하는 검색창의 질의어입니다.

* **Key (K):** "각 항목이 무엇인지 알려주는 색인" 역할로 문서의 제목이나 태그에 해당합니다.

* **Value (V):** "실제로 가져올 내용"인 문서의 본문입니다.
실제 신경망에서는 같은 입력 $X$에 대해 3개의 선형 변환($Q = XW^Q$, $K = XW^K$, $V = XW^V$)을 거쳐 만들어지며, 하나만 뽑는 대신 $Q$와 $K$의 관련도를 바탕으로 V$ 전체를 확률적으로 섞습니다.



**2. Attention Scores**

* **유사도 계산:** $Q$를 모든 $K$와 비교하여 스코어($score(q, k_i)$)를 구하는 단계입니다. 빠르고 메모리 효율적인 내적(Dot-product, $q^Tk$) 방식을 주로 사용하며, Transformer 모델에서는 d_k가 커질 때 내적 분산이 커져 Softmax 기울기가 소실되는 것을 막기 위해 스케일을 보정한 **Scaled dot-product($q^Tk / \sqrt{d_k}$)** 방식을 사용합니다. (이 외에도 Bahdanau의 Additive 방식, Luong의 General 방식 등이 있습니다.)

* **정규화:** 계산된 스코어에 Softmax를 취해 합이 1인 확률 분포($\alpha$)를 만들며, 이것이 Attention Weight가 됩니다. 필요한 경우 패딩이나 미래 토큰의 스코어를 $-\infty$로 채워 가중치를 0으로 만드는 Masking 기법을 적용합니다.



**3. Weighted Sum**
어텐션 가중치($\alpha$)로 $V$를 가중 평균($\sum_i \alpha_i \cdot v_i$)하여 출력하는 3단계 과정으로, 이 "가중 합"이 어텐션의 본질입니다. 일반 신경망의 가중치는 학습 후 고정되지만, 어텐션의 가중치는 입력 데이터 내용에 따라 매번 동적으로 계산된다는 결정적인 차이가 있습니다. 이 전체 과정은 행렬곱 2번과 Softmax 연산으로 이루어져 **완전 병렬 계산**이 가능합니다.

**4. Self-Attention**
$Q$, $K$, $V$가 모두 같은 시퀀스에서 나오는 구조로 "문장이 자기 자신을 보는" 어텐션입니다. 각 토큰이 같은 문장 내 다른 모든 토큰과의 관계로 자신을 재표현하며, 문맥상 대명사("it")가 가리키는 대상을 파악하는 등 상호참조 해소에 강력합니다.

* **RNN 대비 장점:** 순차적으로 연산하는 RNN($O(n)$)과 달리, 아무리 멀리 떨어진 두 토큰도 한 번에 연결(경로 길이 $O(1)$)하므로 장거리 의존성 학습에 유리하고 완전 병렬 처리가 가능합니다.


* 다만 계산 및 메모리 복잡도가 $O(n^2 \cdot d)$이므로 시퀀스가 길어질수록 연산 부담이 커지며, 순열 불변(Permutation invariant) 특성이 있어 단어의 순서를 알 수 없으므로 위치 정보(Positional encoding) 주입이 반드시 필요합니다. 이를 발전시켜 $Q$, $K$, $V$를 여러 개($h$개)로 나누어 서로 다른 표현 부분공간의 관계를 동시에 포착하는 Multi-Head Attention을 활용합니다.

### #2 Transformer 논문 핵심 내용 리뷰

**1. Attention is All You Need**
기존 RNN은 은닉 상태가 순차적으로 의존($h_t = f(h_{t-1}, x_t)$)하여 학습 병렬화가 불가능했고, CNN 계열은 거리에 비례하여 연결 경로 길이가 늘어나 장거리 의존성 포착이 어려웠습니다. 이 논문은 RNN과 CNN을 완전히 배제하고, 두 단어 사이의 경로 길이와 순차 연산을 모두 상수 $O(1)$로 단축시킨 'Transformer' 구조를 제안했습니다. 인코더와 디코더 모두 6개의 층으로 구성되며, 각 서브층마다 잔차 연결과 층 정규화를 적용하여 안정적이고 효율적인 학습을 도모했습니다.

**2. Self-Attention**

* **Scaled Dot-Product Attention:** 내적 차원($d_k$)이 커질수록 분산이 커져 Softmax 기울기가 소실되는 현상을 막기 위해 $\sqrt{d_k}$로 나누어 스케일을 보정합니다 (`Attention(Q,K,V) = softmax(QKᵀ / √d_k) V`).


* **Multi-Head Attention:** $Q, K, V$를 여러 부분공간(논문에서는 8개 헤드)으로 사영하여 병렬 연산함으로써, 문법적 관계나 상호참조 등 **서로 다른 종류의 관계를 동시에 포착**할 수 있도록 설계되었습니다.


* 또한 디코더에서는 미래 위치의 스코어를 $-\infty$로 가리는 Masked Attention을 적용하여 모델의 자기회귀 성질을 보존합니다.



**3. Positional Encoding**
Self-Attention 자체는 단어의 입력 순서를 전혀 알지 못하는 순열 불변의 특성이 있으므로 위치 정보를 명시적으로 주입해야 합니다. 이를 위해 주기적인 **사인 및 코사인 함수**를 사용하여 생성한 위치 정보를 입력 임베딩 벡터에 더해 줍니다. 이 함수를 선택한 핵심 이유는 상대적 위치를 선형 함수로 쉽게 표현할 수 있어 모델이 학습하기 쉽고, 학습 과정에서 보지 못한 **더 긴 시퀀스가 입력되더라도 효과적으로 외삽이 가능**하기 때문입니다.

**4. Parallel Processing**
RNN이 가진 시간축의 순차적 연산 한계를 극복하고, 한 층 내의 모든 위치를 GPU 행렬곱을 통해 한 번에 완전 병렬 계산합니다. 이를 통해 기존 SOTA 모델 대비 **학습 비용을 $\frac{1}{4}$ 이하로 획기적으로 절감**하며 막대한 속도 향상을 이루어냈습니다(Base 모델 12시간, Big 모델 3.5일 소요). 

※ 단, 학습 시에는 병렬 처리가 가능하지만, 실제 추론 시에는 여전히 토큰 단위로 순차적인 처리가 이루어진다는 한계점도 유의해야 합니다.